# 04 - Analisis Sentimen per Aspek Layanan

Notebook ini menjawab pertanyaan inti proyek:

> **Apa yang sudah baik di Silol Kopi & Eatery, dan apa yang masih kurang?**

Label sentimen tetap berasal dari **teks ulasan** (bukan rating), tetapi satu
ulasan sekarang boleh menghasilkan beberapa baris sentimen — satu baris untuk
setiap aspek yang disebut.

## Mengapa harus per aspek?

Ulasan berikut mendapat 5 bintang:

> "Suasana ok, makanan tidak terlalu ok, pelayanan kurang memuaskan"

- Label per ulasan → Positif (atau Netral) → **dua keluhan hilang**.
- Label per aspek → Suasana **Positif**, Makanan **Negatif**, Pelayanan **Negatif**.

## Bagaimana aspek ditentukan? (otomatis)

1. Hitung frekuensi kata pada seluruh ulasan yang telah dipreprocessing.
2. Ambil kata dengan frekuensi ≥ **ambang** (default 100, dapat diubah).
3. Saring: buang stopword, kata sentimen, kata generik, dan kata Inggris —
   agar yang tersisa benar-benar kata benda/aspek layanan.
4. Gabungkan varian kata dengan stemming (`makanan`, `makanannya` → `makan`).
5. Kelompokkan kata kunci yang tersisa dengan *hierarchical clustering*
   berdasarkan co-occurrence.
6. Beri nama tiap kelompok dari aspek acuan domain; kelompok yang tidak cocok
   menjadi aspek baru yang dinamai dari kata tersering.

In [1]:
import sys
from pathlib import Path

# Menemukan root proyek agar `import src` berjalan dari mana pun notebook dibuka.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

print("Root proyek:", ROOT)

Root proyek: D:\hikam\tugas DataMining


## 4.1 Memuat data dan lexicon

In [2]:
from src import aspects, io_utils, lexicon, sentiment

df = io_utils.load_csv(config.PREP_CSV)
lex = lexicon.build_domain_lexicon()
print(f"Ulasan: {len(df):,} | lexicon: {len(lex):,} kata")

Ulasan: 6,795 | lexicon: 7,783 kata


## 4.2 Frekuensi kata: mengapa perlu penyaringan?

Tanpa penyaringan, kata terbanyak didominasi kata sentimen (`enak`, `nyaman`),
kata umum (`nya`, `aja`, `banget`), dan kata Inggris (`the`, `and`). Kata-kata
itu tidak bisa menjadi nama aspek layanan, sehingga harus disaring lebih dulu.

In [3]:
raw_freq = aspects.keyword_frequency(df["tokens_stemmed"])
freq_table = pd.DataFrame(
    {
        "Ambang Frekuensi": [50, 80, 100, 150, 200, 300, 400],
        "Jumlah Kata Lolos Ambang": [int((raw_freq >= t).sum()) for t in [50, 80, 100, 150, 200, 300, 400]],
    }
)
io_utils.save_csv(freq_table, config.TABLES_DIR / "04_ambang_frekuensi.csv")
freq_table

,Ambang Frekuensi,Jumlah Kata Lolos Ambang
0,50,129
1,80,79
2,100,66
3,150,46
4,200,35
5,300,24
6,400,19


In [4]:
raw_freq.head(40).to_frame("Frekuensi (sebelum disaring)")

,Frekuensi (sebelum disaring)
makan,1592
musik,1540
nongkrong,1499
live,1339
enak,1305
harga,938
nyaman,872
layan,849
kopi,782
menu,710


## 4.3 Menjalankan ekstraksi aspek otomatis

**Ambang frekuensi dapat diubah** lewat `MIN_FREQ` di bawah. Semakin kecil
ambang, semakin banyak kata kunci dan aspek yang terbentuk; semakin besar
ambang, semakin ringkas tetapi ada aspek yang mungkin tidak terdeteksi.

In [5]:
MIN_FREQ = config.ASPECT_MIN_FREQ
print(f"Ambang frekuensi yang dipakai: {MIN_FREQ}")

result = aspects.run_aspect_analysis(df, lex, min_freq=MIN_FREQ)
model = result["model"]

print(f"Kata kunci lolos saringan : {len(model.keywords)}")
print(f"Aspek terbentuk           : {len(result['aspects'])}")

Ambang frekuensi yang dipakai: 100


Kata kunci lolos saringan : 39
Aspek terbentuk           : 9


## 4.4 Pengaruh ambang frekuensi

Bandingkan jumlah aspek yang terbentuk pada beberapa ambang. Tabel ini bisa
dipakai untuk memilih ambang yang paling masuk akal untuk laporan.

In [6]:
sweep = []
for th in [50, 100, 150, 200]:
    try:
        m = aspects.build_aspect_model(df["tokens_stemmed"], lex, min_freq=th)
        a_df, _ = aspects.aspect_model_frames(m)
        sweep.append(
            {
                "Ambang": th,
                "Kata Kunci": len(m.keywords),
                "Aspek Terbentuk": len(a_df),
                "Daftar Aspek": ", ".join(a_df["Aspek"].tolist()),
            }
        )
    except ValueError as exc:
        sweep.append({"Ambang": th, "Kata Kunci": 0, "Aspek Terbentuk": 0, "Daftar Aspek": str(exc)})

sweep_df = pd.DataFrame(sweep)
io_utils.save_csv(sweep_df, config.TABLES_DIR / "04_sweep_ambang.csv")
sweep_df

,Ambang,Kata Kunci,Aspek Terbentuk,Daftar Aspek
0,50,67,9,"Suasana, Makanan, Harga, Minuman, Pelayanan, Parkir, Fasilitas, Kecepatan, Kebersihan"
1,100,39,9,"Suasana, Makanan, Harga, Minuman, Pelayanan, Parkir, Kecepatan, Fasilitas, Kebersihan"
2,150,28,9,"Suasana, Makanan, Harga, Minuman, Pelayanan, Kecepatan, Parkir, Fasilitas, Kebersihan"
3,200,23,8,"Suasana, Makanan, Harga, Minuman, Pelayanan, Kecepatan, Parkir, Fasilitas"


## 4.5 Aspek yang terdeteksi dan kata kuncinya

In [7]:
aspects_df, keywords_df = result["aspects"], result["keywords"]
io_utils.save_csv(aspects_df, config.TABLES_DIR / "04_aspek_terdeteksi.csv")
io_utils.save_csv(keywords_df, config.ASPECT_KEYWORD_CSV)
aspects_df

,Aspek,Jumlah Kata Kunci,Kata Kunci,Frekuensi Total
0,Suasana,6,"musik, nongkrong, live, nyaman, ramai, asik",6265
1,Makanan,3,"makan, menu, goreng",2425
2,Harga,4,"harga, jangkau, murah, mahal",1668
3,Minuman,2,"kopi, minum",1365
4,Pelayanan,3,"layan, ramah, staf",1260
5,Parkir,5,"parkir, lokasi, strategis, mobil, motor",922
6,Kecepatan,2,"pesan, cepat",738
7,Fasilitas,4,"lantai, ruang, meja, wifi",685
8,Kebersihan,1,bersih,169


In [8]:
keywords_df.head(40)

,Kata Kunci,Frekuensi,Aspek
0,makan,1592,Makanan
1,musik,1540,Suasana
2,nongkrong,1499,Suasana
3,live,1339,Suasana
4,harga,938,Harga
5,nyaman,872,Suasana
6,layan,849,Pelayanan
7,kopi,782,Minuman
8,menu,710,Makanan
9,minum,583,Minuman


## 4.6 Volume penyebutan per aspek

Menunjukkan aspek mana yang paling sering dibicarakan pelanggan.

In [9]:
summary = result["summary"]
io_utils.save_csv(summary, config.ASPECT_SUMMARY_CSV)
io_utils.save_csv(result["hits"], config.ASPECT_SENTIMENT_CSV)

from src import visualize

visualize.plot_aspect_volume(summary)
summary

,Aspek,Jumlah Disebut,Ulasan Unik,% dari Ulasan,Positif,Netral,Negatif,% Positif,% Negatif,Rating Rata-rata,Hidden Complaint,Skor Prioritas,Peringkat Prioritas,Rekomendasi
0,Kecepatan,690,539,7.93,215,81,394,31.16,57.10,3.02,116,14.999,1,Prioritas 1 - Perbaikan mendesak
1,Makanan,2244,1789,26.33,1526,332,386,68.00,17.20,3.81,189,8.148,2,Prioritas 1 - Perbaikan mendesak
2,Minuman,1299,1066,15.69,691,384,224,53.19,17.24,4.07,117,6.214,3,Prioritas 1 - Perbaikan mendesak
3,Fasilitas,627,463,6.81,221,252,154,35.25,24.56,4.11,105,6.150,4,Prioritas 2 - Perlu ditingkatkan
4,Pelayanan,1010,844,12.42,698,140,172,69.11,17.03,3.30,33,5.412,5,Prioritas 2 - Perlu ditingkatkan
5,Parkir,712,553,8.14,424,146,142,59.55,19.94,4.06,81,5.321,6,Prioritas 2 - Perlu ditingkatkan
6,Suasana,4240,3335,49.08,3469,465,306,81.82,7.22,4.48,234,4.701,7,Pertahankan
7,Harga,1207,1123,16.53,1011,70,126,83.76,10.44,4.31,76,3.627,8,Prioritas 3 - Pantau
8,Kebersihan,137,125,1.84,122,6,9,89.05,6.57,4.20,4,0.769,9,Pertahankan


## 4.7 Komposisi sentimen per aspek

Inilah tabel inti untuk pengambilan keputusan: **aspek mana yang banyak
dikeluhkan dan aspek mana yang menjadi kekuatan**.

In [10]:
summary[[
    "Aspek", "Jumlah Disebut", "Ulasan Unik", "% dari Ulasan",
    "Positif", "Netral", "Negatif", "% Positif", "% Negatif",
    "Rating Rata-rata", "Hidden Complaint",
]]

,Aspek,Jumlah Disebut,Ulasan Unik,% dari Ulasan,Positif,Netral,Negatif,% Positif,% Negatif,Rating Rata-rata,Hidden Complaint
0,Kecepatan,690,539,7.93,215,81,394,31.16,57.10,3.02,116
1,Makanan,2244,1789,26.33,1526,332,386,68.00,17.20,3.81,189
2,Minuman,1299,1066,15.69,691,384,224,53.19,17.24,4.07,117
3,Fasilitas,627,463,6.81,221,252,154,35.25,24.56,4.11,105
4,Pelayanan,1010,844,12.42,698,140,172,69.11,17.03,3.30,33
5,Parkir,712,553,8.14,424,146,142,59.55,19.94,4.06,81
6,Suasana,4240,3335,49.08,3469,465,306,81.82,7.22,4.48,234
7,Harga,1207,1123,16.53,1011,70,126,83.76,10.44,4.31,76
8,Kebersihan,137,125,1.84,122,6,9,89.05,6.57,4.20,4


In [11]:
from src import visualize

visualize.plot_aspect_sentiment(summary)

WindowsPath('D:/hikam/tugas DataMining/results/figures/04_sentimen_aspek.png')

In [12]:
visualize.plot_rating_by_aspect(result["hits"])

WindowsPath('D:/hikam/tugas DataMining/results/figures/04_rating_per_aspek.png')

## 4.8 Matriks prioritas perbaikan

Prioritas dihitung dari **% negatif** dikali **akar volume penyebutan**, sehingga
aspek yang sering dibahas *dan* banyak dikeluhkan menempati peringkat atas.
Kolom *hidden complaint* menunjukkan keluhan dari pelanggan yang tetap memberi
rating 4-5 bintang.

In [13]:
from src import interpret

priority = interpret.priority_matrix(summary)
io_utils.save_csv(priority, config.TABLES_DIR / "04_matriks_prioritas.csv")
priority

,Aspek,Jumlah Disebut,% Negatif,% Positif,Rating Rata-rata,Hidden Complaint,Skor Prioritas,Peringkat Prioritas,Rekomendasi,Kuadran
0,Kecepatan,690,57.10,31.16,3.02,116,14.999,1,Prioritas 1 - Perbaikan mendesak,Temuan Khusus (jarang dibahas tapi negatif)
1,Makanan,2244,17.20,68.00,3.81,189,8.148,2,Prioritas 1 - Perbaikan mendesak,Prioritas Utama (sering dibahas & banyak negatif)
2,Minuman,1299,17.24,53.19,4.07,117,6.214,3,Prioritas 1 - Perbaikan mendesak,Prioritas Utama (sering dibahas & banyak negatif)
3,Fasilitas,627,24.56,35.25,4.11,105,6.150,4,Prioritas 2 - Perlu ditingkatkan,Temuan Khusus (jarang dibahas tapi negatif)
4,Pelayanan,1010,17.03,69.11,3.30,33,5.412,5,Prioritas 2 - Perlu ditingkatkan,Kekuatan (sering dipuji)
5,Parkir,712,19.94,59.55,4.06,81,5.321,6,Prioritas 2 - Perlu ditingkatkan,Temuan Khusus (jarang dibahas tapi negatif)
6,Suasana,4240,7.22,81.82,4.48,234,4.701,7,Pertahankan,Kekuatan (sering dipuji)
7,Harga,1207,10.44,83.76,4.31,76,3.627,8,Prioritas 3 - Pantau,Kekuatan (sering dipuji)
8,Kebersihan,137,6.57,89.05,4.20,4,0.769,9,Pertahankan,"Minor (jarang dibahas, sedikit negatif)"


In [14]:
visualize.plot_priority_scatter(summary)

WindowsPath('D:/hikam/tugas DataMining/results/figures/04_matriks_prioritas.png')

## 4.9 Bukti kualitatif: contoh klausa per aspek

Angka saja tidak cukup untuk meyakinkan pengelola. Berikut kutipan asli
pelanggan untuk setiap aspek.

In [15]:
examples = result["examples"]
io_utils.save_csv(examples, config.TABLES_DIR / "04_contoh_aspek.csv")

neg_examples = examples[examples["Jenis"] == "Negatif"]
for aspect in priority["Aspek"].head(5):
    rows = neg_examples[neg_examples["Aspek"] == aspect].head(3)
    if rows.empty:
        continue
    print(f"\n### {aspect}")
    for _, r in rows.iterrows():
        print(f"  [{r['Rating']}★ | skor {r['Skor']}] {r['Klausa']}")


### Kecepatan
  [1★ | skor -13.2] menunggu satu pesanan lama sekali dari kafe sepi sampai ramai tamunya
  [1★ | skor -13.0] sistem pesan dan bayarnya sangat ribet
  [4★ | skor -12.5] pesan strawberry tea berasa pesan teh tawar  tidak ada manis sama sekali

### Makanan
  [4★ | skor -15.0] kasian meja sebelah sudah menunggu lama dan menu yang dipesan soldout telat dikabari
  [3★ | skor -11.0] minimal goreng nya yang kering biar bisa crispy misal tidak dibuat krispi minimal jangan hancur tahunya   beet thai salad beef nya keras kalii lah
  [3★ | skor -10.0] makannya tidak enak pelayanan agak lama hingar bingar banget karena untuk "konser" musik yang deket banget sama meja karena memang indoor

### Minuman
  [5★ | skor -14.0] non alkohol untuk yang mencari ketenangan saat minum kopi kafe ini tak rekomendasi
  [1★ | skor -9.5] hanya jualan tempat pelayanan buruk rasa makanan buruk waktu menunggu makanan dan minuman sangat lama
  [2★ | skor -9.0] pelayanan yang agak cukup lama di sini saya 

In [16]:
pos_examples = examples[examples["Jenis"] == "Positif"]
for aspect in priority["Aspek"].head(5):
    rows = pos_examples[pos_examples["Aspek"] == aspect].head(2)
    if rows.empty:
        continue
    print(f"\n### {aspect}")
    for _, r in rows.iterrows():
        print(f"  [{r['Rating']}★ | skor {r['Skor']}] {r['Klausa']}")


### Kecepatan
  [5★ | skor 22.0] enak tenang cepat penyajiannya pelayanan bagus puas makan minum disini
  [3★ | skor 18.0] untuk pelayanan cepat dan ramah  suasana juga enak untuk nongkrong di tengah kota jogja

### Makanan
  [5★ | skor 20.0] tempat nongkrong di jogja suka ada live musik makanan enak harga bersahabat di dompet
  [5★ | skor 19.0] dan menarik pelayanan baik dan ramah menu cukup menarik dan enak saya datang waktu sore hari menjelang petang

### Minuman
  [2★ | skor 16.0] saya pikir tempat segede begini kualitas makanan n minuman sudah terjamin enak
  [5★ | skor 15.0] nyaman tempat asik untuk nongkrong makanan minuman enak harga bersahabat

### Fasilitas
  [4★ | skor 17.0] tmp luas parkir enak banyak pilihan meja unik  ada live musik ada tmp sholat banyak spot foto pesan kopi smoothies makan cemilan kopi nya kurang mantap smoothies strawberry
  [5★ | skor 16.2] suasana nyaman sekali dengan desain ruangan yang menarik

### Pelayanan
  [5★ | skor 19.2] tmpat paling nyaman u

## 4.10 Konsistensi sentimen aspek terhadap rating

In [17]:
from src import evaluate

consistency = evaluate.aspect_rating_consistency(result["hits"])
io_utils.save_csv(consistency, config.TABLES_DIR / "04_konsistensi_aspek_rating.csv")
consistency

,Aspek,Sentimen,Rating Rata-rata,Jumlah
0,Fasilitas,Negatif,3.86,154
1,Fasilitas,Netral,4.15,252
2,Fasilitas,Positif,4.24,221
3,Harga,Negatif,3.70,126
4,Harga,Netral,3.70,70
5,Harga,Positif,4.43,1011
6,Kebersihan,Negatif,3.33,9
7,Kebersihan,Netral,2.83,6
8,Kebersihan,Positif,4.33,122
9,Kecepatan,Negatif,2.63,394


## 4.11 Kesimpulan sementara Tahap 4

- Aspek layanan berhasil diekstraksi **otomatis** dari frekuensi kata, dengan
  ambang yang dapat diubah sesuai kebutuhan analisis.
- Setiap aspek memiliki polaritas sendiri sehingga keluhan dan pujian dalam
  satu ulasan tidak saling menutupi.
- Peringkat prioritas menunjukkan urutan aspek yang perlu ditangani lebih dulu.